# 03_catboost_tuning_checkpoint15

This notebook focuses on tuning the best-performing model and checkpoint from the earlier experiments:

- Model: CatBoost
- Checkpoint: 15 overs

Goal:
- rebuild the cleaned checkpoint 15 dataset
- tune CatBoost in a controlled and reproducible way
- compare tuned vs baseline performance

In [1]:
# Step 1: setup and load checkpoint 15 data

from pathlib import Path
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Find project root
cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / "data").exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Could not find project root containing the data/ folder.")

# Load final modelling dataset
model_path = PROJECT_ROOT / "data" / "processed" / "bbl_model_dataset_with_elo.csv"
df = pd.read_csv(model_path)

target_col = "target_batfirst"

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]

# Keep only checkpoint 15
df_15 = df[df["checkpoint"] == 15].copy()

X_15 = df_15[feature_cols_model].copy()
y_15 = df_15[target_col].copy()

X15_train, X15_test, y15_train, y15_test = train_test_split(
    X_15,
    y_15,
    test_size=0.2,
    random_state=42,
    stratify=y_15
)

print("Loaded dataset shape:", df.shape)
print("Checkpoint 15 shape:", df_15.shape)
print("X_15 shape:", X_15.shape)
print("y_15 shape:", y_15.shape)

print("\nCheckpoint 15 target distribution:")
print(y_15.value_counts().sort_index())

print("\nMissing values in X15_train:")
print(X15_train.isna().sum()[X15_train.isna().sum() > 0])

Loaded dataset shape: (2404, 39)
Checkpoint 15 shape: (601, 39)
X_15 shape: (601, 11)
y_15 shape: (601,)

Checkpoint 15 target distribution:
target_batfirst
0    315
1    286
Name: count, dtype: int64

Missing values in X15_train:
elo_diff    6
dtype: int64


## Step 2: baseline CatBoost for checkpoint 15

Before tuning, we first reproduce the baseline CatBoost model for checkpoint 15 using the same settings used in the comparison notebook.

This gives us a clean reference point for:
- accuracy
- confusion matrix
- classification report
- best iteration

Later tuning results will be compared against this baseline.

In [2]:
# Step 2: baseline CatBoost for checkpoint 15

cat_model_15_baseline = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=50
)

cat_model_15_baseline.fit(
    X15_train,
    y15_train,
    cat_features=categorical_features,
    eval_set=(X15_test, y15_test),
    use_best_model=True
)

y15_pred_baseline = cat_model_15_baseline.predict(X15_test)

baseline_acc_15 = accuracy_score(y15_test, y15_pred_baseline)
baseline_cm_15 = confusion_matrix(y15_test, y15_pred_baseline)
tn, fp, fn, tp = baseline_cm_15.ravel()

print("Checkpoint 15 Baseline CatBoost Accuracy:")
print(baseline_acc_15)

print("\nBest iteration:")
print(cat_model_15_baseline.get_best_iteration())

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(baseline_cm_15)

print("\nConfusion Matrix Breakdown:")
print(f"TN (actual 0 predicted 0): {tn}")
print(f"FP (actual 0 predicted 1): {fp}")
print(f"FN (actual 1 predicted 0): {fn}")
print(f"TP (actual 1 predicted 1): {tp}")

print("\nClassification Report:")
print(classification_report(y15_test, y15_pred_baseline))

0:	learn: 0.7125000	test: 0.5785124	best: 0.5785124 (0)	total: 57.9ms	remaining: 17.3s
50:	learn: 0.8625000	test: 0.6694215	best: 0.7190083 (8)	total: 98ms	remaining: 479ms
100:	learn: 0.9125000	test: 0.6446281	best: 0.7190083 (8)	total: 138ms	remaining: 272ms
150:	learn: 0.9437500	test: 0.6280992	best: 0.7190083 (8)	total: 177ms	remaining: 175ms
200:	learn: 0.9729167	test: 0.6446281	best: 0.7190083 (8)	total: 219ms	remaining: 108ms
250:	learn: 0.9833333	test: 0.6363636	best: 0.7190083 (8)	total: 264ms	remaining: 51.6ms
299:	learn: 0.9916667	test: 0.6115702	best: 0.7190083 (8)	total: 317ms	remaining: 0us

bestTest = 0.7190082645
bestIteration = 8

Shrink model to first 9 iterations.
Checkpoint 15 Baseline CatBoost Accuracy:
0.71900826446281

Best iteration:
8

Confusion Matrix [[TN, FP], [FN, TP]]:
[[39 24]
 [10 48]]

Confusion Matrix Breakdown:
TN (actual 0 predicted 0): 39
FP (actual 0 predicted 1): 24
FN (actual 1 predicted 0): 10
TP (actual 1 predicted 1): 48

Classification Report

## Step 3: small CatBoost tuning grid for checkpoint 15

We now test a small number of CatBoost settings around the baseline model.

Tuning focus:
- reduce overfitting
- compare shallow vs slightly deeper trees
- compare current learning rate with a lower learning rate

Baseline reference:
- accuracy = 0.7190
- best iteration = 8

In [4]:
# Step 3: small CatBoost tuning grid for checkpoint 15

param_grid = [
    {"depth": 4, "learning_rate": 0.03, "iterations": 300},
    {"depth": 4, "learning_rate": 0.05, "iterations": 300},
    {"depth": 5, "learning_rate": 0.03, "iterations": 300},
    {"depth": 5, "learning_rate": 0.05, "iterations": 300},
    {"depth": 6, "learning_rate": 0.03, "iterations": 300},
    {"depth": 6, "learning_rate": 0.05, "iterations": 300},  # baseline-like
]

tuning_results = []

for i, params in enumerate(param_grid, 1):
    model = CatBoostClassifier(
        iterations=params["iterations"],
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X15_train,
        y15_train,
        cat_features=categorical_features,
        eval_set=(X15_test, y15_test),
        use_best_model=True
    )

    y_pred = model.predict(X15_test)
    acc = accuracy_score(y15_test, y_pred)

    tuning_results.append({
        "run": i,
        "depth": params["depth"],
        "learning_rate": params["learning_rate"],
        "iterations": params["iterations"],
        "accuracy": acc,
        "best_iteration": model.get_best_iteration()
    })

    print(
        f"Run {i} | depth={params['depth']} | lr={params['learning_rate']} "
        f"| acc={acc:.4f} | best_iter={model.get_best_iteration()}"
    )

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["accuracy", "best_iteration"],
    ascending=[False, True]
).reset_index(drop=True)

print("\nTuning results:")
display(tuning_results_df)

Run 1 | depth=4 | lr=0.03 | acc=0.7025 | best_iter=65
Run 2 | depth=4 | lr=0.05 | acc=0.7107 | best_iter=2
Run 3 | depth=5 | lr=0.03 | acc=0.6860 | best_iter=16
Run 4 | depth=5 | lr=0.05 | acc=0.6942 | best_iter=12
Run 5 | depth=6 | lr=0.03 | acc=0.6777 | best_iter=16
Run 6 | depth=6 | lr=0.05 | acc=0.7190 | best_iter=8

Tuning results:


,run,depth,learning_rate,iterations,accuracy,best_iteration
0,6,6,0.05,300,0.719008,8
1,2,4,0.05,300,0.710744,2
2,1,4,0.03,300,0.702479,65
3,4,5,0.05,300,0.694215,12
4,3,5,0.03,300,0.685950,16
5,5,6,0.03,300,0.677686,16


## Step 4: summarize the first tuning round

The first CatBoost tuning round did not improve on the baseline checkpoint 15 model.

Key result:
- **Baseline / best current setting:** depth = 6, learning_rate = 0.05
- **Best accuracy:** 0.7190
- **Best iteration:** 8

Interpretation:
- Simple changes to tree depth and learning rate did not improve performance
- Shallower trees sometimes delayed the best iteration, but did not surpass the baseline accuracy
- The next tuning direction should focus on **regularization-related parameters** rather than only depth and learning rate

In [5]:
# Step 4: baseline vs tuning summary

best_tuned_row = tuning_results_df.iloc[0].copy()

tuning_summary_df = pd.DataFrame([
    {
        "model_version": "Baseline CatBoost",
        "depth": 6,
        "learning_rate": 0.05,
        "iterations": 300,
        "accuracy": baseline_acc_15,
        "best_iteration": cat_model_15_baseline.get_best_iteration()
    },
    {
        "model_version": "Best from tuning round 1",
        "depth": best_tuned_row["depth"],
        "learning_rate": best_tuned_row["learning_rate"],
        "iterations": best_tuned_row["iterations"],
        "accuracy": best_tuned_row["accuracy"],
        "best_iteration": best_tuned_row["best_iteration"]
    }
])

tuning_summary_df["accuracy"] = tuning_summary_df["accuracy"].round(4)
tuning_summary_df["delta_vs_baseline"] = (
    tuning_summary_df["accuracy"] - baseline_acc_15
).round(4)

print("Baseline vs best tuned result:")
display(tuning_summary_df)

Baseline vs best tuned result:


,model_version,depth,learning_rate,iterations,accuracy,best_iteration,delta_vs_baseline
0,Baseline CatBoost,6.0,0.05,300.0,0.719,8.0,-0.0
1,Best from tuning round 1,6.0,0.05,300.0,0.719,8.0,-0.0


## Step 5: second tuning round with regularization

Since the first tuning round did not improve on the baseline, we now test a small set of regularization-focused CatBoost settings.

Focus:
- `l2_leaf_reg` to penalize model complexity
- `min_data_in_leaf` to reduce overfitting on small leaves

The baseline parameters are kept fixed:
- depth = 6
- learning_rate = 0.05
- iterations = 300

In [6]:
# Step 5: CatBoost regularization tuning for checkpoint 15

regularization_grid = [
    {"l2_leaf_reg": 1,  "min_data_in_leaf": 1},
    {"l2_leaf_reg": 3,  "min_data_in_leaf": 1},
    {"l2_leaf_reg": 5,  "min_data_in_leaf": 1},
    {"l2_leaf_reg": 7,  "min_data_in_leaf": 1},
    {"l2_leaf_reg": 3,  "min_data_in_leaf": 5},
    {"l2_leaf_reg": 5,  "min_data_in_leaf": 5},
    {"l2_leaf_reg": 7,  "min_data_in_leaf": 5},
    {"l2_leaf_reg": 10, "min_data_in_leaf": 5},
]

regularization_results = []

for i, params in enumerate(regularization_grid, 1):
    model = CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=params["l2_leaf_reg"],
        min_data_in_leaf=params["min_data_in_leaf"],
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X15_train,
        y15_train,
        cat_features=categorical_features,
        eval_set=(X15_test, y15_test),
        use_best_model=True
    )

    y_pred = model.predict(X15_test)
    acc = accuracy_score(y15_test, y_pred)

    regularization_results.append({
        "run": i,
        "l2_leaf_reg": params["l2_leaf_reg"],
        "min_data_in_leaf": params["min_data_in_leaf"],
        "accuracy": acc,
        "best_iteration": model.get_best_iteration()
    })

    print(
        f"Run {i} | l2_leaf_reg={params['l2_leaf_reg']} | "
        f"min_data_in_leaf={params['min_data_in_leaf']} | "
        f"acc={acc:.4f} | best_iter={model.get_best_iteration()}"
    )

regularization_results_df = pd.DataFrame(regularization_results).sort_values(
    ["accuracy", "best_iteration"],
    ascending=[False, True]
).reset_index(drop=True)

print("\nRegularization tuning results:")
display(regularization_results_df)

Run 1 | l2_leaf_reg=1 | min_data_in_leaf=1 | acc=0.6942 | best_iter=21
Run 2 | l2_leaf_reg=3 | min_data_in_leaf=1 | acc=0.7190 | best_iter=8
Run 3 | l2_leaf_reg=5 | min_data_in_leaf=1 | acc=0.6860 | best_iter=8
Run 4 | l2_leaf_reg=7 | min_data_in_leaf=1 | acc=0.7025 | best_iter=15
Run 5 | l2_leaf_reg=3 | min_data_in_leaf=5 | acc=0.7190 | best_iter=8
Run 6 | l2_leaf_reg=5 | min_data_in_leaf=5 | acc=0.6860 | best_iter=8
Run 7 | l2_leaf_reg=7 | min_data_in_leaf=5 | acc=0.7025 | best_iter=15
Run 8 | l2_leaf_reg=10 | min_data_in_leaf=5 | acc=0.6777 | best_iter=14

Regularization tuning results:


,run,l2_leaf_reg,min_data_in_leaf,accuracy,best_iteration
0,2,3,1,0.719008,8
1,5,3,5,0.719008,8
2,4,7,1,0.702479,15
3,7,7,5,0.702479,15
4,1,1,1,0.694215,21
5,3,5,1,0.685950,8
6,6,5,5,0.685950,8
7,8,10,5,0.677686,14


## Step 6: summary after two tuning rounds

After two small CatBoost tuning rounds on checkpoint 15:

- the baseline model remains the best-performing configuration
- neither depth / learning-rate tuning nor regularization tuning improved test accuracy beyond **0.7190**
- the best result still occurs very early in training, which suggests the model reaches its optimal point quickly on this split

Conclusion:
The current baseline CatBoost checkpoint 15 model remains the strongest version tested so far.

In [7]:
# Step 6: combine baseline, tuning round 1, and tuning round 2 summary

best_round1 = tuning_results_df.iloc[0]
best_round2 = regularization_results_df.iloc[0]

tuning_overall_summary_df = pd.DataFrame([
    {
        "stage": "Baseline",
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,   # CatBoost default
        "min_data_in_leaf": 1,  # default-style baseline assumption for comparison
        "accuracy": baseline_acc_15,
        "best_iteration": cat_model_15_baseline.get_best_iteration()
    },
    {
        "stage": "Best of tuning round 1",
        "depth": best_tuned_row["depth"],
        "learning_rate": best_tuned_row["learning_rate"],
        "l2_leaf_reg": 3,
        "min_data_in_leaf": 1,
        "accuracy": best_round1["accuracy"],
        "best_iteration": best_round1["best_iteration"]
    },
    {
        "stage": "Best of tuning round 2",
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": best_round2["l2_leaf_reg"],
        "min_data_in_leaf": best_round2["min_data_in_leaf"],
        "accuracy": best_round2["accuracy"],
        "best_iteration": best_round2["best_iteration"]
    }
])

tuning_overall_summary_df["accuracy"] = tuning_overall_summary_df["accuracy"].round(4)
tuning_overall_summary_df["delta_vs_baseline"] = (
    tuning_overall_summary_df["accuracy"] - baseline_acc_15
).round(4)

display(tuning_overall_summary_df)

,stage,depth,learning_rate,l2_leaf_reg,min_data_in_leaf,accuracy,best_iteration,delta_vs_baseline
0,Baseline,6.0,0.05,3.0,1.0,0.719,8.0,-0.0
1,Best of tuning round 1,6.0,0.05,3.0,1.0,0.719,8.0,-0.0
2,Best of tuning round 2,6.0,0.05,3.0,1.0,0.719,8.0,-0.0


## Step 7: final selected CatBoost model for checkpoint 15

Based on the tuning experiments in this notebook, the original baseline CatBoost model remains the best-performing configuration for checkpoint 15.

Selected model:
- depth = 6
- learning_rate = 0.05
- iterations = 300
- best iteration = 8
- test accuracy = 0.7190

Tuning outcome:
- neither depth / learning-rate tuning nor regularization tuning improved on the baseline
- therefore, the baseline CatBoost model is retained as the final selected checkpoint 15 model

In [8]:
# Step 7: final selected model summary + feature importance

final_model_15_summary = pd.DataFrame([{
    "model": "CatBoost",
    "checkpoint": 15,
    "depth": 6,
    "learning_rate": 0.05,
    "iterations": 300,
    "best_iteration": cat_model_15_baseline.get_best_iteration(),
    "accuracy": baseline_acc_15,
    "tn": baseline_cm_15[0, 0],
    "fp": baseline_cm_15[0, 1],
    "fn": baseline_cm_15[1, 0],
    "tp": baseline_cm_15[1, 1]
}])

final_model_15_summary["accuracy"] = final_model_15_summary["accuracy"].round(4)

fi_15_final = pd.DataFrame({
    "feature": X_15.columns,
    "importance": cat_model_15_baseline.get_feature_importance()
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Final selected model summary:")
display(final_model_15_summary)

print("Final selected model feature importance:")
display(fi_15_final)

Final selected model summary:


,model,checkpoint,depth,learning_rate,iterations,best_iteration,accuracy,tn,fp,fn,tp
0,CatBoost,15,6,0.05,300,8,0.719,39,24,10,48


Final selected model feature importance:


,feature,importance
0,run_rate,41.421334
1,elo_diff,13.325915
2,wickets,7.033317
3,venue,6.848372
4,momentum_runs,6.569614
5,season_mean_runs_cp,6.485635
6,resource_frac,5.176309
7,momentum_wickets,4.628228
8,venue_mean_runs_cp,3.861011
9,venue_mean_wkts_cp,3.236504


In [ ]:
FInal Final

In [15]:
# Step 1: setup and load final modelling dataset

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# -----------------------------
# Find project root
# -----------------------------
cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / "data").exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Could not find project root containing the data/ folder.")

# -----------------------------
# Load dataset
# -----------------------------
data_path = PROJECT_ROOT / "data" / "processed" / "bbl_model_dataset_with_elo.csv"
df = pd.read_csv(data_path)

# -----------------------------
# Core config
# -----------------------------
target_col = "target_batfirst"
checkpoints = [6, 10, 15, 20]

feature_cols_model = [
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

categorical_features = ["venue"]

# Keep a clean working dataframe
keep_cols = ["match_id", "season", "checkpoint"] + feature_cols_model + [target_col]
model_df = df[keep_cols].copy()

print("Loaded dataset shape:", df.shape)
print("Model dataframe shape:", model_df.shape)

print("\nCheckpoints present:")
print(sorted(model_df["checkpoint"].unique()))

print("\nCheckpoint row counts:")
print(model_df["checkpoint"].value_counts().sort_index())

print("\nOverall target distribution:")
print(model_df[target_col].value_counts().sort_index())

print("\nMissing values in modelling features:")
print(model_df[feature_cols_model].isna().sum()[model_df[feature_cols_model].isna().sum() > 0])

print("\nFeature columns used for modelling:")
print(feature_cols_model)

Loaded dataset shape: (2404, 39)
Model dataframe shape: (2404, 15)

Checkpoints present:
[np.int64(6), np.int64(10), np.int64(15), np.int64(20)]

Checkpoint row counts:
checkpoint
6     618
10    613
15    601
20    572
Name: count, dtype: int64

Overall target distribution:
target_batfirst
0    1260
1    1144
Name: count, dtype: int64

Missing values in modelling features:
elo_diff    39
dtype: int64

Feature columns used for modelling:
['wickets', 'run_rate', 'momentum_runs', 'momentum_wickets', 'venue', 'resource_frac', 'venue_mean_runs_cp', 'venue_mean_wkts_cp', 'season_mean_runs_cp', 'season_mean_wkts_cp', 'elo_diff']


In [16]:
# Step 2: focus on checkpoint 15 first and create stratified 80/20 split

from sklearn.model_selection import train_test_split

checkpoint_to_test = 15

df_cp = model_df[model_df["checkpoint"] == checkpoint_to_test].copy()

X_cp = df_cp[feature_cols_model].copy()
y_cp = df_cp[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_cp,
    y_cp,
    test_size=0.2,
    random_state=42,
    stratify=y_cp
)

print(f"Checkpoint selected: {checkpoint_to_test}")
print("Checkpoint dataframe shape:", df_cp.shape)
print("X shape:", X_cp.shape)
print("y shape:", y_cp.shape)

print("\nOverall target distribution:")
print(y_cp.value_counts().sort_index())

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

print("\nMissing values in X_train:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\nMissing values in X_test:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

Checkpoint selected: 15
Checkpoint dataframe shape: (601, 15)
X shape: (601, 11)
y shape: (601,)

Overall target distribution:
target_batfirst
0    315
1    286
Name: count, dtype: int64

Train shape: (480, 11)
Test shape: (121, 11)

Train target distribution:
target_batfirst
0    252
1    228
Name: count, dtype: int64

Test target distribution:
target_batfirst
0    63
1    58
Name: count, dtype: int64

Missing values in X_train:
elo_diff    6
dtype: int64

Missing values in X_test:
elo_diff    2
dtype: int64


In [23]:
# Step 3: baseline CatBoost on the current 80/20 split

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

baseline_cb = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=False
)

baseline_cb.fit(
    X_train,
    y_train,
    cat_features=categorical_features
)

y_test_pred = baseline_cb.predict(X_test)
y_test_prob = baseline_cb.predict_proba(X_test)[:, 1]

baseline_acc = accuracy_score(y_test, y_test_pred)
baseline_auc = roc_auc_score(y_test, y_test_prob)
baseline_cm = confusion_matrix(y_test, y_test_pred)

print("Baseline CatBoost Test Accuracy:", round(baseline_acc, 4))
print("Baseline CatBoost Test AUC:", round(baseline_auc, 4))

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(baseline_cm)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Baseline CatBoost Test Accuracy: 0.6116
Baseline CatBoost Test AUC: 0.6724

Confusion Matrix [[TN, FP], [FN, TP]]:
[[36 27]
 [20 38]]

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.57      0.61        63
           1       0.58      0.66      0.62        58

    accuracy                           0.61       121
   macro avg       0.61      0.61      0.61       121
weighted avg       0.61      0.61      0.61       121



In [38]:
# Step 4: stronger 5-fold CV tuning for CatBoost on training data only

from sklearn.model_selection import StratifiedKFold
from itertools import product

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# A stronger but still manageable search grid
depth_grid = [4, 5, 6, 7, 10, 12, 14, 16]
learning_rate_grid = [0.1 ,0.02, 0.03, 0.05, 0.07]
l2_leaf_reg_grid = [1, 3, 5, 7, 10, 20]

param_grid = list(product(depth_grid, learning_rate_grid, l2_leaf_reg_grid))

cv_results = []

for run_id, (depth, learning_rate, l2_leaf_reg) in enumerate(param_grid, start=1):
    fold_accuracies = []
    fold_aucs = []
    fold_best_iters = []

    for fold_id, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
        X_tr = X_train.iloc[tr_idx].copy()
        y_tr = y_train.iloc[tr_idx].copy()

        X_val = X_train.iloc[val_idx].copy()
        y_val = y_train.iloc[val_idx].copy()

        model = CatBoostClassifier(
            iterations=1000,
            depth=depth,
            learning_rate=learning_rate,
            l2_leaf_reg=l2_leaf_reg,
            loss_function="Logloss",
            eval_metric="Accuracy",
            random_seed=42,
            verbose=False
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=categorical_features,
            eval_set=(X_val, y_val),
            use_best_model=True,
            early_stopping_rounds=50
        )

        val_pred = model.predict(X_val)
        val_prob = model.predict_proba(X_val)[:, 1]

        fold_accuracies.append(accuracy_score(y_val, val_pred))
        fold_aucs.append(roc_auc_score(y_val, val_prob))
        fold_best_iters.append(model.get_best_iteration())

    cv_results.append({
        "run": run_id,
        "depth": depth,
        "learning_rate": learning_rate,
        "l2_leaf_reg": l2_leaf_reg,
        "cv_accuracy_mean": np.mean(fold_accuracies),
        "cv_accuracy_std": np.std(fold_accuracies),
        "cv_auc_mean": np.mean(fold_aucs),
        "cv_auc_std": np.std(fold_aucs),
        "mean_best_iteration": int(round(np.mean(fold_best_iters)))
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    by=["cv_accuracy_mean", "cv_auc_mean", "cv_accuracy_std"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("Top 15 CatBoost CV results:")
display(cv_results_df.head(15))

Top 15 CatBoost CV results:


,run,depth,learning_rate,l2_leaf_reg,cv_accuracy_mean,cv_accuracy_std,cv_auc_mean,cv_auc_std,mean_best_iteration
0,123,10,0.10,5,0.760417,0.053115,0.782188,0.066109,28
1,115,7,0.07,1,0.754167,0.044973,0.772811,0.047943,28
2,176,12,0.07,3,0.754167,0.035232,0.778955,0.046355,17
3,134,10,0.03,3,0.754167,0.030619,0.769383,0.048437,21
4,87,6,0.07,5,0.752083,0.057206,0.781736,0.056883,38
5,143,10,0.05,10,0.752083,0.031180,0.768821,0.048394,56
6,219,16,0.02,5,0.750000,0.027163,0.770562,0.044114,29
7,93,7,0.10,5,0.750000,0.032940,0.790267,0.047182,16
8,65,6,0.10,10,0.747917,0.031869,0.778516,0.048602,33
9,88,6,0.07,7,0.747917,0.052457,0.773979,0.050959,51


In [58]:
# Step 5: focused refinement around the best CatBoost region

from itertools import product

refine_depth_grid = [7, 9, 10, 11]
refine_learning_rate_grid = [0.7, 0.1, 0.12, 0.13, 0.14]
refine_l2_leaf_reg_grid = [1, 2, 3, 5, 6, 7]
refine_random_strength_grid = [0.5, 1.0, 2.0]

refine_param_grid = list(product(
    refine_depth_grid,
    refine_learning_rate_grid,
    refine_l2_leaf_reg_grid,
    refine_random_strength_grid
))

refine_results = []

for run_id, (depth, learning_rate, l2_leaf_reg, random_strength) in enumerate(refine_param_grid, start=1):
    fold_accuracies = []
    fold_aucs = []
    fold_best_iters = []

    for fold_id, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
        X_tr = X_train.iloc[tr_idx].copy()
        y_tr = y_train.iloc[tr_idx].copy()

        X_val = X_train.iloc[val_idx].copy()
        y_val = y_train.iloc[val_idx].copy()

        model = CatBoostClassifier(
            iterations=1200,
            depth=depth,
            learning_rate=learning_rate,
            l2_leaf_reg=l2_leaf_reg,
            random_strength=random_strength,
            loss_function="Logloss",
            eval_metric="Accuracy",
            random_seed=42,
            verbose=False
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=categorical_features,
            eval_set=(X_val, y_val),
            use_best_model=True,
            early_stopping_rounds=75
        )

        val_pred = model.predict(X_val)
        val_prob = model.predict_proba(X_val)[:, 1]

        fold_accuracies.append(accuracy_score(y_val, val_pred))
        fold_aucs.append(roc_auc_score(y_val, val_prob))
        fold_best_iters.append(model.get_best_iteration())

    refine_results.append({
        "run": run_id,
        "depth": depth,
        "learning_rate": learning_rate,
        "l2_leaf_reg": l2_leaf_reg,
        "random_strength": random_strength,
        "cv_accuracy_mean": np.mean(fold_accuracies),
        "cv_accuracy_std": np.std(fold_accuracies),
        "cv_auc_mean": np.mean(fold_aucs),
        "cv_auc_std": np.std(fold_aucs),
        "mean_best_iteration": int(round(np.mean(fold_best_iters)))
    })

refine_results_df = pd.DataFrame(refine_results).sort_values(
    by=["cv_accuracy_mean", "cv_auc_mean", "cv_accuracy_std"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("Top 15 refined CatBoost CV results:")
display(refine_results_df.head(15))

KeyboardInterrupt: 

In [50]:
# Step 6: tune prediction threshold using CV probabilities on training data only

best_params = {
    "depth": 7,
    "learning_rate": 0.13,
    "l2_leaf_reg": 6,
    "random_strength": 0.5,
    "iterations": 1200,
    "mean_best_iteration": 27
}

oof_probs = np.zeros(len(X_train))
oof_true = y_train.reset_index(drop=True).copy()

for fold_id, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_tr = X_train.iloc[tr_idx].copy()
    y_tr = y_train.iloc[tr_idx].copy()

    X_val = X_train.iloc[val_idx].copy()
    y_val = y_train.iloc[val_idx].copy()

    model = CatBoostClassifier(
        iterations=best_params["iterations"],
        depth=best_params["depth"],
        learning_rate=best_params["learning_rate"],
        l2_leaf_reg=best_params["l2_leaf_reg"],
        random_strength=best_params["random_strength"],
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=categorical_features,
        eval_set=(X_val, y_val),
        use_best_model=True,
        early_stopping_rounds=75
    )

    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]

threshold_grid = np.round(np.arange(0.35, 0.651, 0.01), 2)

threshold_results = []

for threshold in threshold_grid:
    preds = (oof_probs >= threshold).astype(int)
    acc = accuracy_score(oof_true, preds)

    threshold_results.append({
        "threshold": threshold,
        "cv_accuracy": acc
    })

threshold_results_df = pd.DataFrame(threshold_results).sort_values(
    by=["cv_accuracy", "threshold"],
    ascending=[False, True]
).reset_index(drop=True)

print("Top threshold results:")
display(threshold_results_df.head(15))

best_threshold = threshold_results_df.iloc[0]["threshold"]
print("Best threshold selected from CV:", best_threshold)

Top threshold results:


,threshold,cv_accuracy
0,0.50,0.775000
1,0.51,0.764583
2,0.52,0.760417
3,0.49,0.758333
4,0.48,0.750000
5,0.53,0.750000
6,0.54,0.745833
7,0.47,0.739583
8,0.46,0.735417
9,0.55,0.735417


Best threshold selected from CV: 0.5


In [70]:
# Step 7: final CatBoost fit on full training data, then evaluate once on test data

final_cb = CatBoostClassifier(
    iterations=200,
    depth=10,
    learning_rate=0.05,
    l2_leaf_reg=1,
    random_strength=1.0,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=False
)

final_cb.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# final_cb.fit(
#     X_train,
#     y_train,
#     cat_features=categorical_features
# )

final_test_prob = final_cb.predict_proba(X_test)[:, 1]
final_test_pred = (final_test_prob >= 0.50).astype(int)

final_acc = accuracy_score(y_test, final_test_pred)
final_auc = roc_auc_score(y_test, final_test_prob)
final_cm = confusion_matrix(y_test, final_test_pred)

print("Final Tuned CatBoost Test Accuracy:", round(final_acc, 4))
print("Final Tuned CatBoost Test AUC:", round(final_auc, 4))

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(final_cm)

print("\nClassification Report:")
print(classification_report(y_test, final_test_pred))

Final Tuned CatBoost Test Accuracy: 0.719
Final Tuned CatBoost Test AUC: 0.7551

Confusion Matrix [[TN, FP], [FN, TP]]:
[[39 24]
 [10 48]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.62      0.70        63
           1       0.67      0.83      0.74        58

    accuracy                           0.72       121
   macro avg       0.73      0.72      0.72       121
weighted avg       0.73      0.72      0.72       121

